# 03 - Cleaning, FK Validation and Quarantine

Cleans each dataset, validates **all primary-key and foreign-key
relationships** across datasets, quarantines problematic records
(never silently dropped), and writes the processed layer:

- order lines must belong to the order's own restaurant menu
- orders must reference known customers / restaurants / promotions
- promotions must be active on the order date at that restaurant
- ratings must come from the order's customer and rate an item in the order
- inventory / wastage must reference items owned by the restaurant
- order dates must fall inside the 12-month analysis period

Every removal is recorded in `cleaning_log.csv`.


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import nb_common

# ------------------------------------------------------------------
# Run mode:
#   "quick" -> demo-scale data (~20k orders) in notebook/outputs/
#   "full"  -> full SRS scale (1M lines) in the canonical repo dirs
# ------------------------------------------------------------------
MODE = "quick"

paths = nb_common.setup_paths(MODE)
nb_common.ensure_config(paths)
mk = nb_common.markers(paths)

pd.options.display.max_columns = 20
print(f"MODE: {MODE}")
print(f"raw       : {paths['raw']}")
print(f"processed : {paths['processed']}")


MODE: quick
raw       : /home/user/Techwizz/notebook/outputs/quick/raw
processed : /home/user/Techwizz/notebook/outputs/quick/processed


In [2]:
if not nb_common.step_done(*mk['generation']):
    from generate_dineiq_data import generate
    generate(raw_dir=paths['raw'], config_path=paths['config'])
if not nb_common.step_done(*mk['quality']):
    import data_quality_check
    data_quality_check.main(raw_dir=paths['raw'], report_dir=paths['reports'])

In [3]:
import clean_dineiq_data

if not nb_common.step_done(*mk['cleaning']):
    clean_dineiq_data.main(
        raw_dir=paths['raw'],
        processed_dir=paths['processed'],
        reports_dir=paths['reports'],
        quarantine_dir=paths['quarantine'],
        config_path=paths['config'],
    )
else:
    print('Processed layer already exists - skipping (delete', paths['processed'], 'to rerun).')

DineIQ Analytics - Data Cleaning Pipeline
Raw data       : /home/user/Techwizz/notebook/outputs/quick/raw
Processed data : /home/user/Techwizz/notebook/outputs/quick/processed
Quarantine     : /home/user/Techwizz/notebook/outputs/quick/reports/quarantine


Cleaning: locations.csv
  Input rows: 20
  Output rows: 20

Cleaning: restaurants.csv
  Input rows: 20
  Output rows: 20

Cleaning: menu_categories.csv
  Input rows: 10
  Output rows: 10

Cleaning: menu_items.csv
  Input rows: 150
  Output rows: 150

Cleaning: customers.csv
  Input rows: 8,040
  Duplicate customer rows removed: 40
  Missing emails retained: 80
  Output rows: 8,000



Cleaning: orders.csv
  Input rows: 20,000
  Missing payment methods: 199
  Total calculation mismatches: 0
  Output rows: 19,980



Cleaning: order_items.csv
  Input rows: 200,040
  Output rows: 199,440

Cleaning: pricing_history.csv
  Input rows: 272
  Output rows: 272

Cleaning: promotions.csv
  Input rows: 120
  Output rows: 120



Cleaning: ratings.csv
  Input rows: 20,000
  Output rows: 19,868

Cleaning: inventory.csv
  Input rows: 10,000
  Stock arithmetic mismatches: 0
  Output rows: 10,000

Cleaning: wastage.csv
  Input rows: 10,000
  Output rows: 10,000



DATA CLEANING COMPLETE
Processed data : /home/user/Techwizz/notebook/outputs/quick/processed
Cleaning log   : /home/user/Techwizz/notebook/outputs/quick/reports/cleaning_log.csv
Summary report : /home/user/Techwizz/notebook/outputs/quick/reports/cleaning_summary.csv
Quarantine     : /home/user/Techwizz/notebook/outputs/quick/reports/quarantine

            dataset  raw_rows  processed_rows  rows_removed_or_quarantined
      locations.csv        20              20                            0
    restaurants.csv        20              20                            0
menu_categories.csv        10              10                            0
     menu_items.csv       150             150                            0
      customers.csv      8040            8000                           40
         orders.csv     20000           19980                           20
    order_items.csv    200040          199440                          600
pricing_history.csv       272             272       

In [4]:
print('--- Cleaning summary (raw -> processed) ---')
pd.read_csv(paths['reports'] / 'cleaning_summary.csv')

--- Cleaning summary (raw -> processed) ---


,dataset,raw_rows,processed_rows,rows_removed_or_quarantined
0,locations.csv,20,20,0
1,restaurants.csv,20,20,0
2,menu_categories.csv,10,10,0
3,menu_items.csv,150,150,0
4,customers.csv,8040,8000,40
5,orders.csv,20000,19980,20
6,order_items.csv,200040,199440,600
7,pricing_history.csv,272,272,0
8,promotions.csv,120,120,0
9,ratings.csv,20000,19868,132


In [5]:
print('--- Quarantine files and reasons ---')
rows = []
for f in sorted(paths['quarantine'].glob('*.csv')):
    q = pd.read_csv(f, low_memory=False)
    if 'quarantine_reason' in q.columns:
        for reason, n in q['quarantine_reason'].value_counts().items():
            rows.append({'file': f.name, 'reason': reason, 'rows': int(n)})
pd.DataFrame(rows)

--- Quarantine files and reasons ---


,file,reason,rows
0,customers_quarantine.csv,Duplicate customer_id; first occurrence retained.,40
1,order_items_quarantine.csv,"Invalid order item key, quantity, price, disco...",400
2,order_items_quarantine.csv,Unknown order_id,200
3,orders_quarantine.csv,order_date outside analysis period (2025-01-01...,20
4,ratings_quarantine.csv,Rated menu item is not part of the referenced ...,113
5,ratings_quarantine.csv,Unknown order_id,19


In [6]:
# Post-cleaning verification: zero orphans must remain in the processed layer
proc = paths['processed']
od = pd.read_csv(proc / 'orders.csv', low_memory=False)
oi = pd.read_csv(proc / 'order_items.csv', low_memory=False)
mi = pd.read_csv(proc / 'menu_items.csv', low_memory=False)
cu = pd.read_csv(proc / 'customers.csv', low_memory=False)
ra = pd.read_csv(proc / 'ratings.csv', low_memory=False)

checks = {
    'orders -> customers': int((~od['customer_id'].isin(set(cu['customer_id']))).sum()),
    'order items -> orders': int((~oi['order_id'].isin(set(od['order_id']))).sum()),
    'order items restaurant consistency': int(
        (oi.merge(od[['order_id', 'restaurant_id']], on='order_id')
         .pipe(lambda m: (m['menu_item_id'].map(mi.set_index('menu_item_id')['restaurant_id']) != m['restaurant_id'])))
        .sum()),
}
lines = oi.groupby('order_id')['menu_item_id'].apply(set).to_dict()
checks['ratings item-in-order orphans'] = int(
    sum(1 for r in ra.itertuples() if r.menu_item_id not in lines.get(r.order_id, set()))
)
res = pd.Series(checks)
assert (res == 0).all(), f'FK violations remain: {res[res != 0].to_dict()}'
print('All FK checks pass - zero orphans in the processed layer.')
res

All FK checks pass - zero orphans in the processed layer.


orders -> customers                   0
order items -> orders                 0
order items restaurant consistency    0
ratings item-in-order orphans         0
dtype: int64

In [7]:
print('--- FK validation entries from the cleaning log ---')
log = pd.read_csv(paths['reports'] / 'cleaning_log.csv', low_memory=False)
log[log['action'].astype(str).str.startswith('fk_validation_')]

--- FK validation entries from the cleaning log ---


,run_time,dataset,action,affected_rows,reason
1,2026-09-24 07:39:37,restaurants.csv,fk_validation_location_id,0,Restaurants must reference a known location.
4,2026-09-24 07:39:37,menu_items.csv,fk_validation_restaurant_category,0,Menu items must reference known restaurants an...
11,2026-09-24 07:39:37,orders.csv,fk_validation_customer_id,0,Orders must reference a known customer.
12,2026-09-24 07:39:37,orders.csv,fk_validation_restaurant_id,0,Orders must reference a known restaurant.
14,2026-09-24 07:39:37,orders.csv,fk_validation_analysis_period,20,Order dates must fall inside the 12-month anal...
17,2026-09-24 07:39:37,order_items.csv,fk_validation_order_id,200,Order items must reference a known order.
18,2026-09-24 07:39:37,order_items.csv,fk_validation_menu_item_id,0,Order items must reference a known menu item.
19,2026-09-24 07:39:37,order_items.csv,fk_validation_restaurant_consistency,0,Order items must come from the order's own res...
23,2026-09-24 07:39:37,pricing_history.csv,fk_validation_menu_item_id,0,Pricing history must reference a known menu item.
25,2026-09-24 07:39:37,promotions.csv,fk_validation_menu_item_id,0,Promotions must reference a known menu item.


**Next:** `04_processing_and_integration.ipynb` - integrate the
cleaned datasets and build the core analytics.
